In [2]:
import pandas as pd
import os

In [89]:
def remove_outliers_iqr(df):
    df_cleaned = df.copy()

    for column in df_cleaned.columns:
        if pd.api.types.is_numeric_dtype(df_cleaned[column]):
            Q1 = df_cleaned[column].quantile(0.25)
            Q3 = df_cleaned[column].quantile(0.75)
            
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            df_cleaned = df_cleaned[(df_cleaned[column] >= lower_bound) & (df_cleaned[column] <= upper_bound)]
    
    return df_cleaned

In [3]:
labels_df = pd.read_csv(r"PHQ8_labels.csv")

In [95]:

participants = list(labels_df["Participant_ID"])
#participants = [367, 368]
print(participants)

[300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 395, 396, 397, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492]


In [91]:
display(labels_df[labels_df["Participant_ID"] == 376])

,Participant_ID,PHQ8_Binary,PHQ8_Score
75,376,1,12


In [92]:
dir_path = r"datasets\DAIC-Woz"
output_path = r"audio_csv_withoutliers"

In [93]:
complete_audio_woOutliers = pd.DataFrame()

for participant in participants:
    print(participant)

    folder_path = fr"{participant}_P"
    temp_covarep = pd.read_csv(os.path.join(dir_path, folder_path, fr"{participant}_COVAREP.csv"), header = None)
    
    temp_formant = pd.read_csv(os.path.join(dir_path, folder_path, fr"{participant}_FORMANT.csv"), header = None)
    #renaming formant columns
    temp_formant.columns = ["F0", "F1", "F2", "F3", "F4"]

    temp = pd.concat([temp_formant, temp_covarep], axis = 1)
    temp.dropna(inplace = True, axis = 0)
    # droping VUV = 0 , non vocal rows
    temp.drop(temp.loc[temp[1]==0].index, inplace=True)
    temp.reset_index(drop = True, inplace = True)
    temp.drop(columns=[1], inplace = True, axis = 1)

    # removinf outliers
    temp = remove_outliers_iqr(temp)

    # adding target label
    temp["Target"] = labels_df[labels_df["Participant_ID"] == participant].iloc[0,1]

    # saving csv per participant
    temp.to_csv(os.path.join(output_path, fr"{participant}_withoutlier.csv"), index = False)

    complete_audio_woOutliers = pd.concat([complete_audio_woOutliers, temp], axis = 0)
    print(complete_audio_woOutliers.shape)

367
(71983, 79)
368
(148722, 79)


In [81]:
complete_audio_woOutliers.reset_index(inplace = True, drop = True)
display(complete_audio_woOutliers)

,F0,F1,F2,F3,F4,0,2,3,4,5,...,65,66,67,68,69,70,71,72,73,Target
0,293.20,1635.5,2538.1,3242.0,3988.0,269.0,0.038446,0.111760,0.9758,0.22440,...,-0.92749,-0.74712,-0.54045,-0.42579,-0.40818,-0.27205,-0.18758,-0.115740,-0.079553,0
1,318.92,1656.0,2726.0,3413.0,4225.6,269.0,0.025810,0.080351,0.0000,0.34022,...,-0.93171,-0.74674,-0.55854,-0.43645,-0.41822,-0.27231,-0.18013,-0.141710,-0.093744,0
2,582.25,1469.5,2423.0,3034.3,4043.2,271.0,0.045581,0.179910,-5.3092,0.47652,...,-0.98807,-0.77188,-0.59350,-0.41623,-0.29127,-0.19868,-0.11678,-0.124110,-0.113760,0
3,476.97,2103.9,3067.2,4082.9,4639.4,231.0,0.058752,0.170800,-4.2914,0.26402,...,-0.66118,-0.66544,-0.68464,-0.62956,-0.61106,-0.35705,-0.26494,-0.145970,-0.166570,0
4,507.50,1305.0,1740.2,2812.2,3542.5,182.0,0.082644,0.237140,-4.6301,0.67685,...,-0.70488,-0.71061,-0.69635,-0.60842,-0.58210,-0.39583,-0.28554,-0.204080,-0.150490,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
678383,222.24,1945.2,2819.0,4040.8,4784.6,152.0,0.201670,0.548830,9.2067,0.35271,...,-0.88044,-0.56210,-0.64258,-0.57587,-0.43868,-0.30497,-0.28417,-0.144310,-0.051550,0
678384,581.16,1820.0,2496.1,3225.3,4017.1,209.0,0.034666,0.349750,5.6566,0.12914,...,-0.84567,-0.73813,-0.63178,-0.54548,-0.51871,-0.30890,-0.27262,-0.134150,-0.067753,0
678385,523.00,1251.3,2007.7,2864.9,3914.0,207.5,0.134000,0.386820,-5.1442,0.19695,...,-0.83250,-0.73273,-0.62752,-0.53681,-0.51673,-0.27603,-0.26620,-0.111950,-0.042231,0
678386,523.00,852.0,1765.2,2886.5,4060.9,206.0,0.162830,0.502370,10.1350,0.21850,...,-0.86057,-0.74837,-0.61421,-0.49968,-0.47533,-0.19490,-0.19090,-0.020043,0.059151,0


In [78]:
complete_audio_woOutliers.to_csv(r"complete_audio_woOutliers.csv", index = False)